# Cross-Attention text-conditioned Pokémon DDPM —— two-stage (exp07)

Compared to exp06, adds **cross-attention**: lets each pixel of the image "query" **every token** of the text,
Achieve **spatialized text control** (no longer just one global text vector added uniformly across the whole image). This is what makes it possible to "put the right feature in the right place".

**Two conditioning paths:**
- Time + **pooled text vector** → added globally into ResBlock (controls the whole)
- **Text token sequence** → cross-attn (24², 12² layers) → spatialized control

CLIP's per-token features are saved offline as `clip_seq.pt`, and training loads them directly without touching CLIP.
Two-stage + rehearsal, same as exp06. (Does not touch any of your existing notebooks.)

## Model (with SelfAttn + CrossAttn)

In [1]:
import math, os, glob, json, random
import torch
import torch.nn as nn
import torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"

def sinusoidal_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, c_dim, groups=8):
        super().__init__()
        self.block1 = nn.Sequential(nn.GroupNorm(groups, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.cond_mlp = nn.Sequential(nn.SiLU(), nn.Linear(c_dim, out_ch))
        self.block2 = nn.Sequential(nn.GroupNorm(groups, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, c):
        h = self.block1(x); h = h + self.cond_mlp(c)[:, :, None, None]; h = self.block2(h)
        return h + self.skip(x)

class SelfAttn(nn.Module):
    """pixel <-> pixel self-attention (handles global coherence)."""
    def __init__(self, ch, heads=4):
        super().__init__(); self.norm = nn.GroupNorm(8, ch); self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x):
        B, C, H, W = x.shape
        h = self.norm(x).reshape(B, C, H*W).transpose(1, 2); h, _ = self.mha(h, h, h)
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class CrossAttn(nn.Module):
    """pixel <-> text token cross-attention (spatialized text control): each pixel "queries" every word of the text."""
    def __init__(self, ch, ctx_dim=512, heads=4):
        super().__init__(); self.norm = nn.GroupNorm(8, ch); self.kv = nn.Linear(ctx_dim, ch)
        self.mha = nn.MultiheadAttention(ch, heads, batch_first=True)
    def forward(self, x, ctx, key_padding_mask=None):
        B, C, H, W = x.shape
        q = self.norm(x).reshape(B, C, H*W).transpose(1, 2)   # (B, HW, C) image as query
        kv = self.kv(ctx)                                     # (B, L, C) text tokens as key/value
        h, _ = self.mha(q, kv, kv, key_padding_mask=key_padding_mask)  # mask: True=padding ignored
        return x + h.transpose(1, 2).reshape(B, C, H, W)

class Stage(nn.Module):
    """n_blocks ResBlocks + optional (self-attn -> cross-attn)."""
    def __init__(self, in_ch, out_ch, c_dim, n_blocks=2, attn=False):
        super().__init__()
        self.blocks = nn.ModuleList([ResBlock(in_ch if i==0 else out_ch, out_ch, c_dim) for i in range(n_blocks)])
        self.self_attn  = SelfAttn(out_ch)  if attn else None
        self.cross_attn = CrossAttn(out_ch) if attn else None
    def forward(self, x, c, ctx=None, mask=None):
        for b in self.blocks: x = b(x, c)
        if self.self_attn  is not None: x = self.self_attn(x)
        if self.cross_attn is not None: x = self.cross_attn(x, ctx, mask)
        return x

class UNet(nn.Module):
    """Text-conditioned UNet. Two condition paths: time + pooled text -> global add (ResBlock); text token sequence -> cross-attn (spatial)."""
    def __init__(self, in_ch=3, base=128, c_dim=256, n_blocks=2, txt_dim=512):
        super().__init__(); self.c_dim = c_dim
        self.text_proj = nn.Sequential(nn.Linear(txt_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))  # pooled -> global condition
        self.time_mlp  = nn.Sequential(nn.Linear(c_dim, c_dim), nn.SiLU(), nn.Linear(c_dim, c_dim))
        self.in_conv = nn.Conv2d(in_ch, base, 3, padding=1)
        self.down1 = Stage(base,    base,   c_dim, n_blocks)
        self.down2 = Stage(base,    base*2, c_dim, n_blocks)
        self.down3 = Stage(base*2,  base*4, c_dim, n_blocks, attn=True)
        self.downsample = nn.AvgPool2d(2)
        self.mid = Stage(base*4, base*4, c_dim, n_blocks, attn=True)
        self.upsample = nn.Upsample(scale_factor=2, mode="nearest")
        self.up3 = Stage(base*4+base*4, base*2, c_dim, n_blocks, attn=True)
        self.up2 = Stage(base*2+base*2, base,   c_dim, n_blocks)
        self.up1 = Stage(base+base,     base,   c_dim, n_blocks)
        self.out = nn.Sequential(nn.GroupNorm(8, base), nn.SiLU(), nn.Conv2d(base, in_ch, 3, padding=1))
    def forward(self, x, t, pooled, tokens, mask):
        c = self.time_mlp(sinusoidal_embedding(t, self.c_dim)) + self.text_proj(pooled)   # time + pooled text (global)
        x = self.in_conv(x)
        s1 = self.down1(x, c); x = self.downsample(s1)
        s2 = self.down2(x, c); x = self.downsample(s2)
        s3 = self.down3(x, c, tokens, mask); x = self.downsample(s3)            # cross-attn @24
        x = self.mid(x, c, tokens, mask)                                       # cross-attn @12
        x = self.upsample(x); x = self.up3(torch.cat([x, s3], 1), c, tokens, mask)  # cross-attn @24
        x = self.upsample(x); x = self.up2(torch.cat([x, s2], 1), c)
        x = self.upsample(x); x = self.up1(torch.cat([x, s1], 1), c)
        return self.out(x)

## Diffusion + loss + sampling (sampling includes CFG)

In [2]:
class Diffusion:
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device="cuda"):
        self.T = timesteps; self.device = device
        beta = torch.linspace(beta_start, beta_end, timesteps, device=device)
        self.beta = beta; self.alpha = 1.0 - beta; self.alpha_bar = torch.cumprod(self.alpha, 0)
    def q_sample(self, x0, t, noise):
        ab = self.alpha_bar[t][:, None, None, None]; return ab.sqrt()*x0 + (1-ab).sqrt()*noise
    def loss(self, model, x0, pooled, tokens, mask):
        B = x0.size(0); t = torch.randint(0, self.T, (B,), device=x0.device); noise = torch.randn_like(x0)
        return F.mse_loss(model(self.q_sample(x0, t, noise), t, pooled, tokens, mask), noise)
    @torch.no_grad()
    def sample(self, model, n, pooled, tokens, mask, null, guidance=4.0, img_size=96):
        """pooled(n,512) tokens(n,L,512) mask(n,L); null=(p,t,m) single entry, expanded to n internally."""
        model.eval()
        np_, nt_, nm_ = null
        np_ = np_[None].expand(n, -1); nt_ = nt_[None].expand(n, -1, -1); nm_ = nm_[None].expand(n, -1)
        x = torch.randn(n, 3, img_size, img_size, device=self.device)
        for i in reversed(range(self.T)):
            t = torch.full((n,), i, device=self.device, dtype=torch.long)
            ec = model(x, t, pooled, tokens, mask)             # conditional
            eu = model(x, t, np_, nt_, nm_)                    # unconditional (null)
            pred = eu + guidance * (ec - eu)                   # CFG
            bt, at, abt = self.beta[i], self.alpha[i], self.alpha_bar[i]
            mean = (1/at.sqrt()) * (x - (bt/(1-abt).sqrt()) * pred)
            x = mean + (bt.sqrt()*torch.randn_like(x) if i > 0 else 0.0)
        model.train(); return x

## Experiment / stage configuration (two stages + rehearsal, keep both weights)

In [3]:
STAGE = 1   # ← 1 or 2
EXP_BASE  = "/root/autodl-tmp/experiments"
ALL_DIR   = "/root/autodl-tmp/pokemon_all"
BASE = 128; N_BLOCKS = 2; EMA_DECAY = 0.999
IMG_SIZE = 96; BATCH_SIZE = 64; TIMESTEPS = 1000

if STAGE == 1:
    FOCUS_DIR = ALL_DIR; DATASET_TAG = "pokemonALL"; INIT_CKPT = None
    REHEARSAL_N = 0; EPOCHS = 300; LR = 2e-4
else:
    FOCUS_DIR = "/root/autodl-tmp/pokemon20_aug"; DATASET_TAG = "pokemon20aug"
    INIT_CKPT = os.path.join(EXP_BASE, "exp07_stage1_pokemonALL_xattn/checkpoints/ckpt_ep300.pt")
    REHEARSAL_N = 3000; EPOCHS = 120; LR = 1e-4

EXP_NAME = f"exp07_stage{STAGE}_{DATASET_TAG}_xattn"
OUT_DIR  = os.path.join(EXP_BASE, EXP_NAME)
CKPT_DIR = os.path.join(OUT_DIR,"checkpoints"); SAMPLE_DIR = os.path.join(OUT_DIR,"samples"); TB_DIR = os.path.join(OUT_DIR,"tb")
for d in (CKPT_DIR, SAMPLE_DIR, TB_DIR): os.makedirs(d, exist_ok=True)
try:
    os.makedirs("/root/tf-logs", exist_ok=True); link = os.path.join("/root/tf-logs", EXP_NAME)
    if not os.path.exists(link): os.symlink(TB_DIR, link)
except OSError: pass
import shutil   # auto-archive: copy this script (real filename) into the current experiment directory
SCRIPT_NAME = "pokemon_train_xattn.ipynb"
SCRIPT_HOME = os.path.join(EXP_BASE, "exp07_stage1_pokemonALL_xattn", SCRIPT_NAME)   # the script's home
try:
    dst = os.path.join(OUT_DIR, SCRIPT_NAME)
    if os.path.abspath(SCRIPT_HOME) != os.path.abspath(dst): shutil.copy(SCRIPT_HOME, dst)
except Exception: pass
print("experiment:", EXP_NAME, "| STAGE", STAGE, "| rehearsal", REHEARSAL_N)

实验: exp07_stage1_pokemonALL_xattn | STAGE 1 | 复习 0


## Load per-token text features (training never touches CLIP)

`clip_seq.pt` contains each description's tokens (77×512)/pooled (512)/mask (77). A **null** row (empty string) is appended at the end,
During training, tokens are used as a class table: `POOL[y]/TOK[y]/MASK[y]`; on CFG dropout `y -> null`.

In [4]:
seq = torch.load("/root/autodl-tmp/clip_seq.pt", map_location="cpu")
n2i = {n: i for i, n in enumerate(seq["names"])}
# classes for this stage = full set (consistent across both stages, the rehearsed other Pokémon are also in the table)
names = sorted({os.path.basename(p).split("__")[0] for p in glob.glob(os.path.join(ALL_DIR, "*.png"))} & set(n2i))
idx = [n2i[n] for n in names]
# Append null row -> index NULL used for CFG
TOK  = torch.cat([seq["tokens"][idx],  seq["null_tokens"][None]]).float().to(device)   # (C+1,77,512)
POOL = torch.cat([seq["pooled"][idx],  seq["null_pooled"][None]]).to(device)            # (C+1,512)
MASK = torch.cat([seq["mask"][idx],    seq["null_mask"][None]]).to(device)              # (C+1,77)
NUM_CLASSES = len(names); NULL = NUM_CLASSES
json.dump(names, open(os.path.join(OUT_DIR,"classes.json"),"w",encoding="utf-8"), ensure_ascii=False, indent=1)
print("classes:", NUM_CLASSES, "| TOK", tuple(TOK.shape))

类别: 988 | TOK (989, 77, 512)


## Data (specialization + rehearsal)

In [5]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

c2i = {n: i for i, n in enumerate(names)}
class DS(Dataset):
    def __init__(self, paths):
        self.paths = [p for p in paths if os.path.basename(p).split("__")[0] in c2i]
        self.tf = transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(), transforms.Normalize([0.5]*3,[0.5]*3)])
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]; img = Image.open(p)
        if img.mode in ("RGBA","LA","P"):
            img = Image.alpha_composite(Image.new("RGBA", img.size, (255,)*4), img.convert("RGBA")).convert("RGB")
        else: img = img.convert("RGB")
        return self.tf(img), c2i[os.path.basename(p).split("__")[0]]

focus = glob.glob(os.path.join(FOCUS_DIR, "*.png")); rehe = []
if REHEARSAL_N > 0:
    pool = glob.glob(os.path.join(ALL_DIR, "*.png")); random.seed(0); rehe = random.sample(pool, min(REHEARSAL_N, len(pool)))
ds = DS(focus + rehe)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True, drop_last=True)
print(f"focus {len(focus)} + rehearsal {len(rehe)} = {len(ds)} images")

专精 13085 + 复习 0 = 13085 张


## Training (EMA + CFG dropout + TensorBoard)

CFG dropout is done in the training loop: with 10% probability replace the sample's condition index with `NULL`. Validation images use **pikachu** (in-class)
Makes early progress easier to see.

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import matplotlib.pyplot as plt

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay; self.shadow = {k: v.detach().clone() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self, model, step):
        d = min(self.decay, (1+step)/(10+step))
        for k, v in model.state_dict().items():
            s = self.shadow[k]; s.mul_(d).add_(v.detach(), alpha=1-d) if v.dtype.is_floating_point else s.copy_(v)
    def copy_to(self, model): model.load_state_dict(self.shadow, strict=True)

writer = SummaryWriter(TB_DIR)
model = UNet(base=BASE, n_blocks=N_BLOCKS).to(device)
if INIT_CKPT:
    model.load_state_dict(torch.load(INIT_CKPT, map_location=device)); print("Initialized from Stage1")
ema = EMA(model, EMA_DECAY); ema_model = UNet(base=BASE, n_blocks=N_BLOCKS).to(device)
diff = Diffusion(timesteps=TIMESTEPS, device=device)
opt = torch.optim.AdamW(model.parameters(), lr=LR); scaler = torch.amp.GradScaler("cuda")
print("trainable params: %.2fM" % (sum(p.numel() for p in model.parameters())/1e6))

eval_idx = names.index("pikachu") if "pikachu" in names else 0
null = (POOL[NULL], TOK[NULL], MASK[NULL]); SAMPLE_EVERY = 10

step = 0
for epoch in range(1, EPOCHS+1):
    model.train(); pbar = tqdm(dl, desc=f"[S{STAGE}] ep{epoch}/{EPOCHS}")
    for x0, y in pbar:
        x0 = x0.to(device, non_blocking=True); y = y.to(device, non_blocking=True)
        y = y.clone(); y[torch.rand(y.size(0), device=device) < 0.1] = NULL   # CFG dropout
        opt.zero_grad()
        with torch.amp.autocast("cuda"):
            loss = diff.loss(model, x0, POOL[y], TOK[y], MASK[y])
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        ema.update(model, step); writer.add_scalar("train/loss", loss.item(), step); step += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}")
    if epoch % SAMPLE_EVERY == 0 or epoch == EPOCHS:
        ema.copy_to(ema_model)
        img = (diff.sample(ema_model, 1, POOL[eval_idx:eval_idx+1], TOK[eval_idx:eval_idx+1], MASK[eval_idx:eval_idx+1], null, guidance=3.0, img_size=IMG_SIZE).clamp(-1,1)+1)/2
        plt.figure(figsize=(2.4,2.6)); plt.imshow(img[0].cpu().permute(1,2,0).numpy()); plt.axis("off"); plt.title(names[eval_idx], fontsize=9)
        plt.savefig(os.path.join(SAMPLE_DIR, f"sample_ep{epoch}.png"), dpi=120, bbox_inches="tight")
        writer.add_figure("samples", plt.gcf(), epoch); plt.close()
        torch.save(ema.shadow, os.path.join(CKPT_DIR, f"ckpt_ep{epoch}.pt"))
        print(f"  ✔ ep{epoch} saved -> {EXP_NAME}")
writer.close()

可训练参数: 33.51M


[S1] ep10/300: 100%|██████████| 204/204 [01:19<00:00,  2.57it/s, loss=0.0178]


  ✔ ep10 已存 -> exp07_stage1_pokemonALL_xattn


[S1] ep20/300: 100%|██████████| 204/204 [01:19<00:00,  2.57it/s, loss=0.0150]


  ✔ ep20 已存 -> exp07_stage1_pokemonALL_xattn


...

[S1] ep50/300: 100%|██████████| 204/204 [01:19<00:00,  2.57it/s, loss=0.0139]


  ✔ ep50 已存 -> exp07_stage1_pokemonALL_xattn


[S1] ep60/300: 100%|██████████| 204/204 [01:20<00:00,  2.55it/s, loss=0.0144]


  ✔ ep60 已存 -> exp07_stage1_pokemonALL_xattn


[S1] ep61/300:  35%|███▌      | 72/204 [00:27<00:51,  2.56it/s, loss=0.0087]

## ✅ Next steps
- Stage 1 (`STAGE=1`) retrains from scratch (new architecture, old weights unusable) → then `STAGE=2` fine-tunes.
- Generation/fusion/free text → `pokemon_gen_xattn.ipynb`.